# Oil Futures Calendar Spread — OLS Forecasting Study
**Single-file, executable coursework program** · Historical monthly sample through June 2024

**Question:** Can the current WTI futures curve and a small set of lagged economic fundamentals forecast the **next-month change** in the CL2−CL1 spread?

Run cells in order. No online data, no Brent, no external project `.py` modules. The notebook directly reads the `Oil_Data.csv` file, which contains the four required source columns from the professor-supplied workbook (355 monthly rows). Ensure the `Oil_Data.csv` file is in the same directory as this notebook before execution.

**Important:** CL1/CL2 are continuous futures series. Price differences and trading P&L below are *proxies*, not verified executable profits. Macro publication delays are assumptions, not validated real-time vintages. The 2019–2024 period has previously been examined; redesigns require a new untouched evaluation period.

**Setup:** Use Python 3.10 or later with `numpy`, `pandas`, `matplotlib`, `statsmodels`, `openpyxl`, `ipykernel` and Jupyter installed. From the notebook's Python environment, run `%pip install numpy pandas matplotlib statsmodels openpyxl ipykernel jupyter` once if needed, then restart the kernel.


In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import acorr_breusch_godfrey, het_breuschpagan, het_arch, breaks_cusumolsresid


# Embedded source-data extract is compressed and checked below; original supplied workbook is preferred when available.
SEED = 20261022
FEATURES = ['spread','spread_change','inventory_growth','ip_growth']
OLS_SPECS = {'OLS2': FEATURES[:2], 'OLS3': FEATURES[:3], 'OLS4': FEATURES}
VALID_START, VALID_END, TEST_START = pd.Period('2015-01','M'), pd.Period('2018-12','M'), pd.Period('2019-01','M')

In [2]:
# A compressed CSV extract of ONLY month, CL1, CL2, Inventories and IndustrialProduction Index.
# This was extracted without changing numeric observations from the professor-supplied Oil Data.xlsx.
# This cell contains DATA, not another model or an external download.
import pandas as pd

# Directly read the Excel file
monthly = pd.read_excel('Oil_Data.xlsx', sheet_name='Industry_M')
monthly['month'] = pd.PeriodIndex(monthly['Date'], freq='M')
monthly = monthly.set_index('month')

print('Source months:', len(monthly), monthly.index.min(), 'to', monthly.index.max())


Source months: 355 1994-12 to 2024-06


## 1. Import and clean — monthly observations
We align `Industry_M` and `Macro_M` by **calendar month**, because source dates may be a trading day or calendar month-end. Monthly data align futures prices with macroeconomic observations and support medium-term positioning, but cannot capture short-lived trading opportunities. We do not interpolate missing observations. Undated rows are removed; duplicate months, nonnumeric values and missing required data are checked explicitly. Brent is not selected as a model variable.

A workbook byte-checksum difference triggers a warning rather than automatic rejection: simply re-saving an Excel file can change its checksum. The selected observations must still match the embedded original-data extract within numerical tolerance. A genuinely changed dataset requires an explicit re-audit.


In [4]:
# Directly read the CSV file
monthly = pd.read_excel('Oil_Data.xlsx', sheet_name='Industry_M')
monthly['month'] = pd.PeriodIndex(monthly['month'], freq='M')
monthly = monthly.set_index('month')

# Ensure the data is valid
assert (monthly[['Inventories', 'IndustrialProduction Index']].dropna() > 0).all().all()
assert not monthly.index.has_duplicates
assert monthly.index.equals(pd.period_range(monthly.index.min(), monthly.index.max(), freq='M'))

print('Source months:', len(monthly), monthly.index.min(), 'to', monthly.index.max())
print('Missing selected observations:', monthly.isna().sum().to_dict())
print('Brent used:', False)

assert (monthly[['Inventories','IndustrialProduction Index']].dropna()>0).all().all()
assert not monthly.index.has_duplicates
assert monthly.index.equals(pd.period_range(monthly.index.min(),monthly.index.max(),freq='M'))
print('Source months:',len(monthly),monthly.index.min(),'to',monthly.index.max())
print('Missing selected observations:',monthly.isna().sum().to_dict())
print('Brent used:',False)

KeyError: 'month'

## 2. Define the target and the **four pre-specified** candidate features
The level of the futures calendar spread is \(S_t=CL2_t-CL1_t\) in USD/barrel. The forecast outcome is \(y_{t+1}=S_{t+1}-S_t\), also USD/barrel. Candidate predictors are current spread, previous spread change, **two-month-delayed** log inventory growth and **two-month-delayed** log industrial-production growth. This is a one-month forecast horizon, *not* a two-month forecast.

The growth transformation \(100\log(X_t/X_{t-1})\) accommodates scale differences; **OLS's original-unit predictors are not z-score standardized**. Correlations describe association, not causality.

**Economic motivation:** A calendar-spread change relates directly to the direction of a spread position. The current spread captures potential mean reversion; its recent change captures persistence or reversal. Inventory growth represents changes in physical market availability, while industrial-production growth proxies economic activity and oil demand. These are parsimonious candidate predictors, not established causal drivers. The assumed two-month delays reduce availability concerns but do not reconstruct publication dates or unrevised historical data vintages.


In [ ]:
f=monthly[['CL1','CL2']].copy()
f['spread']=monthly['CL2']-monthly['CL1']
f['spread_change']=f['spread'].diff()
f['inventory_growth']=100*np.log(monthly['Inventories']).diff().shift(2)
f['ip_growth']=100*np.log(monthly['IndustrialProduction Index']).diff().shift(2)
f['target']=(f['spread'].shift(-1)-f['spread'])
f['spread_next']=f['spread'].shift(-1)
f['target_month']=f.index+1
frame=f.dropna(subset=FEATURES+['target','spread_next','CL1','CL2']).copy()
assert not frame.empty
assert (frame['target_month']==frame.index+1).all()
assert np.allclose(frame['target'],frame['spread_next']-frame['spread'])
assert frame.index.equals(pd.period_range(frame.index.min(),frame.index.max(),freq='M'))
print(f'Complete labelled targets: {len(frame)} | development: {(frame.target_month<TEST_START).sum()} | evaluation: {(frame.target_month>=TEST_START).sum()}')
assert len(frame)==351 and (frame.target_month>=TEST_START).sum()==66, 'Dataset differs from validated supplied historical project'
fig,ax=plt.subplots(figsize=(10,3)); monthly.eval('CL2-CL1').plot(ax=ax); ax.axhline(0,color='grey',lw=1)
ax.set(title='Observed CL2 − CL1 calendar spread',ylabel='USD per barrel');plt.show()
print('Development-sample correlations with next-month change:')
dev=frame[frame.target_month<TEST_START]
display(dev[FEATURES+['target']].corr().round(3))

## 3. OLS and chronological one-step forecasts
OLS is estimated *with an intercept*, by minimising squared errors. No predictor uses a future target. For forecast origin \(t\), training includes only outcomes known through \(t\). The model is re-estimated every month. We test 2-, 3- and 4-predictor predeclared specifications, plus zero-change and expanding-mean benchmarks. The main model is chosen **only on 2015–2018 validation**.

In [ ]:
def predict_ols(train, current, predictors):
    X=sm.add_constant(train[predictors],has_constant='add')
    model=sm.OLS(train['target'],X).fit()
    new=sm.add_constant(pd.DataFrame([current[predictors]]),has_constant='add')
    return float(model.predict(new).iloc[0])

def rolling_predictions(frame, model, periods, window=None):
    out=[]
    for idx in periods:
        pos=frame.index.get_loc(idx)
        train=frame.iloc[max(0,pos-window) if window else 0:pos]
        assert len(train)>=60
        assert pd.Period(train['target_month'].iloc[-1],freq='M')<=idx, 'Future target leaked into training'
        current=frame.loc[idx]
        if model in OLS_SPECS:
            yhat=predict_ols(train,current,OLS_SPECS[model])
        elif model=='Zero change': yhat=0.0
        elif model=='Expanding mean': yhat=float(train['target'].mean())
        else: raise ValueError(model)
        out.append({'model':model,'origin':str(idx),'target_month':str(idx+1),'actual':float(current['target']),
                    'prediction':yhat,'spread':float(current['spread']), 'CL1':float(current['CL1']),'CL2':float(current['CL2'])})
    return pd.DataFrame(out)

def score(panel):
    e=panel.actual-panel.prediction
    return {'n':len(e),'RMSE':float(np.sqrt(np.mean(e**2))),'MAE':float(np.mean(abs(e))),
            'MSE':float(np.mean(e**2))}

validation_periods=frame.index[(frame.target_month>=VALID_START)&(frame.target_month<=VALID_END)]
validation={m:rolling_predictions(frame,m,validation_periods) for m in [*OLS_SPECS,'Zero change','Expanding mean']}
valid_scores=pd.DataFrame({m:score(p) for m,p in validation.items()}).T.sort_values('RMSE')
display(valid_scores.round(5))
selected_ols=min(OLS_SPECS,key=lambda m:score(validation[m])['MSE'])
print('Selected OLS from pre-test validation:',selected_ols)
# Retain the validation-selected specification even if a different model wins in evaluation.

## 4. ARMA as a traditional time-series comparator (not a machine-learning model)
A simple ARMA(1,1) is an optional traditional comparison. The stationary **level vs change representation** is selected from pre-2015 observations only (ADF and KPSS). The same forecast origins and target months are used as OLS. If the numerical optimisation fails, the notebook reports it instead of silently substituting a prediction.

In [ ]:
initial=frame[frame.target_month<VALID_START]
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    level_adf=adfuller(initial['spread_next'],maxlag=12,regression='c',autolag='AIC')[1]
    level_kpss=kpss(initial['spread_next'],regression='c',nlags='auto')[1]
arma_representation='level' if level_adf<.05 and level_kpss>=.05 else 'change'
print('ARMA representation:',arma_representation,'; pre-2015 ADF p=',round(level_adf,4),'KPSS p=',round(level_kpss,4))

def rolling_arma(frame,periods,order=(1,1),representation=arma_representation):
    rec=[]
    for period in periods:
        pos=frame.index.get_loc(period); train=frame.iloc[:pos]
        assert train.target_month.iloc[-1]<=period
        series=train['spread_next' if representation=='level' else 'target'].to_numpy(float)
        last_error=None
        for method in ['lbfgs','bfgs']:
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter('ignore')
                    fit=ARIMA(series,order=(order[0],0,order[1]),trend='c',enforce_stationarity=True,enforce_invertibility=True).fit(method_kwargs={'method':method,'maxiter':500,'disp':0})
                if not fit.mle_retvals.get('converged',False):raise RuntimeError('ARMA optimiser did not converge')
                value=float(fit.forecast(1)[0]); break
            except Exception as exc: last_error=exc
        else: raise RuntimeError(f'ARMA failed at {period}: {last_error}')
        row=frame.loc[period]
        forecast=value-float(row['spread']) if representation=='level' else value
        rec.append({'model':'ARMA(1,1)','origin':str(period),'target_month':str(period+1), 'actual':float(row['target']), 'prediction':forecast,'spread':float(row['spread']), 'CL1':float(row['CL1']),'CL2':float(row['CL2'])})
    return pd.DataFrame(rec)

# Fixed optional comparator, not an order selected by the executed validation code.
selected_arma_order=(1,1)
print('ARMA comparison order:',selected_arma_order)

## 5. Later historical forecasts and **fair benchmark comparison**
All models forecast the same 66 target months from January 2019 through June 2024. We compare RMSE and MAE in USD/barrel and \(1-MSE_{model}/MSE_{zero}\). A smaller RMSE does not imply a trading profit.

The default reported comparison contains OLS and the two simple benchmarks. ARMA is excluded unless `INCLUDE_ARMA=True` and all subsequent cells are rerun. This later historical period has previously been examined and must not be described as a completely untouched final holdout.


In [ ]:
test_periods=frame.index[frame.target_month>=TEST_START]
all_models=[*OLS_SPECS,'Zero change','Expanding mean']
test_panels={m:rolling_predictions(frame,m,test_periods) for m in all_models}
# ARMA fitting is comparatively slow because it is re-estimated for every month.
INCLUDE_ARMA=False  # Set True to run the slower secondary ARMA model (optional)
if INCLUDE_ARMA:
    test_panels['ARMA(1,1)']=rolling_arma(frame,test_periods,selected_arma_order)
base_mse=score(test_panels['Zero change'])['MSE']
results=pd.DataFrame({m:{**score(p),'MSE improvement vs zero (%)':100*(1-score(p)['MSE']/base_mse)}
                      for m,p in test_panels.items()}).T.sort_values('RMSE')
display(results[['n','RMSE','MAE','MSE improvement vs zero (%)']].round(4))
assert len(test_panels[selected_ols])==66
fig,ax=plt.subplots(figsize=(10,4))
results['RMSE'].sort_values().plot(kind='barh',ax=ax,color='steelblue');ax.invert_yaxis()
ax.set(xlabel='RMSE (USD/barrel; lower is better)',title='Historical one-month-ahead forecast accuracy')
plt.tight_layout();plt.show()

## 6. Regression interpretation and standard econometric diagnostics
Fit OLS4 only on **pre-2019 development data** for the explanatory coefficient table. The forecast experiment itself refits at every origin. HAC (Newey–West) standard errors account approximately for serial correlation/heteroskedasticity in inference; they do not change OLS coefficients or cure omitted variables. ADF has the unit-root null; KPSS has a stationarity null.

**Stationarity qualification:** For the development sample, ADF rejects a unit root in the spread level, but KPSS rejects level stationarity (p ≈ 0.022). The evidence is mixed, so retaining the spread as a mean-reversion predictor is a modelling assumption rather than an established stationarity result. Instability or regime changes remain possible; the rolling-window check provides only a limited sensitivity assessment. Inventory growth's ADF p-value is about 0.04999, very close to the 5% cutoff, so its classification is sensitive to test settings. The other differenced/growth series show evidence consistent with stationarity under these specifications.


In [ ]:
cols=OLS_SPECS[selected_ols]
X=sm.add_constant(dev[cols],has_constant='add')
ols=sm.OLS(dev.target,X).fit()
robust=ols.get_robustcov_results(cov_type='HAC',maxlags=6)
coef=pd.DataFrame({'coefficient':robust.params,'HAC SE':robust.bse,'HAC p-value':robust.pvalues},index=X.columns)
display(coef.round(5))
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    diagnostics={
        'Breusch–Godfrey p (12)':acorr_breusch_godfrey(ols,nlags=12)[1],
        'Breusch–Pagan p':het_breuschpagan(ols.resid,ols.model.exog)[1],
        'ARCH LM p (6)':het_arch(ols.resid,nlags=6)[1],
        'CUSUM p':breaks_cusumolsresid(ols.resid,ddof=int(ols.df_model)+1)[1],
        'ADF target p':adfuller(dev.target,maxlag=12,autolag='AIC')[1],
        'KPSS target p':kpss(dev.target,nlags='auto')[1],
    }
display(pd.Series(diagnostics,name='p-value').to_frame().round(5))
fig,axes=plt.subplots(1,2,figsize=(11,3.5))
axes[0].scatter(ols.fittedvalues,ols.resid,s=12,alpha=.6);axes[0].axhline(0,color='grey');axes[0].set(xlabel='OLS fitted spread change',ylabel='Residual',title='Residuals vs fitted')
sm.qqplot(ols.resid,line='s',ax=axes[1]);axes[1].set(title='OLS residual Q–Q')
plt.tight_layout();plt.show()
print('Interpret coefficients as conditional associations, not causal effects.')
from statsmodels.tools.sm_exceptions import InterpolationWarning

# Apply the same stationarity checks to every predictor and the dependent variable.
stationarity_rows=[]
for variable in FEATURES+['target']:
    series=dev[variable].dropna()
    adf_result=adfuller(series,maxlag=12,regression='c',autolag='AIC')
    with warnings.catch_warnings(record=True) as recorded:
        warnings.simplefilter('always')
        kpss_result=kpss(series,regression='c',nlags='auto')
    adf_p,kpss_p=adf_result[1],kpss_result[1]
    if adf_p<0.05 and kpss_p>=0.05:
        conclusion='Evidence consistent with stationarity'
    elif adf_p>=0.05 and kpss_p<0.05:
        conclusion='Evidence against stationarity'
    else:
        conclusion='Mixed or inconclusive evidence'
    stationarity_rows.append({'variable':variable,'n':len(series),
        'ADF p-value':adf_p,'KPSS p-value':kpss_p,
        'assessment (5%)':conclusion,
        'KPSS note':'; '.join(str(w.message) for w in recorded if issubclass(w.category, InterpolationWarning))})
stationarity=pd.DataFrame(stationarity_rows).set_index('variable')
print('Development-sample stationarity checks: ADF null = unit root; KPSS null = stationarity.')
display(stationarity.round(5))
print('KPSS boundary p-values are bounds from the tabulated distribution, not exact probabilities.')


## 7. Uncertainty in improvement — paired moving-block bootstrap
A 95% interval is generated by resampling paired monthly squared-loss differences in consecutive six-month circular blocks. This uses the **existing forecasts** and does not account for re-tuning, model search or structural shifts.

In [ ]:
p=test_panels[selected_ols]
base=test_panels['Zero change']
assert p.target_month.tolist()==base.target_month.tolist()
d=((base.actual-base.prediction)**2-(p.actual-p.prediction)**2).to_numpy()
rng=np.random.default_rng(SEED); n=len(d); block=6; reps=2000
boot=np.empty(reps)
for j in range(reps):
    starts=rng.integers(0,n,size=int(np.ceil(n/block)))
    indices=np.concatenate([(s+np.arange(block))%n for s in starts])[:n]
    boot[j]=d[indices].mean()
print('Mean squared-loss benefit (USD/barrel)^2:',round(d.mean(),5))
print('95% moving-block interval:',np.round(np.quantile(boot,[.025,.975]),5))
print('Interval includes zero:',np.quantile(boot,.025)<=0<=np.quantile(boot,.975))

## 8. Trading decision illustration **versus baselines**
**Important limitation:** The following is a *continuous-spread change proxy*, **not verified futures P&L**. The project lacks contract IDs, roll timing, executable bid/ask quotes, margin and collateral returns. If predicted spread change exceeds +$0.10, the illustration is long CL2/short CL1; below −$0.10, the opposite; otherwise flat. Each active month is a newly entered and closed two-leg position, so four one-way transactions are charged. We compare the selected OLS signal, an always-long-spread baseline, and flat exposure. Graphs start at 100 solely for visual comparison, not as investable wealth indices.

In [ ]:
COST_LEG_WAY=0.025  # USD per barrel per leg per one-way transaction; assumption, not observed cost
BAND=0.10
p=test_panels[selected_ols].copy().sort_values('target_month')
actual=p.actual.to_numpy(float)
def proxy_path(signal,name,cost=COST_LEG_WAY):
    q=np.where(signal>BAND,1,np.where(signal< -BAND,-1,0))
    gross=q*actual
    costs=4*cost*(q!=0)
    net=gross-costs
    return pd.DataFrame({'month':p.target_month,'strategy':name,'position':q,'net':net,'cum_dollars':np.cumsum(net),'index_100':100+np.cumsum(net)})
long_short=proxy_path(p.prediction.to_numpy(float),selected_ols)
always_long=proxy_path(np.full(len(p),np.inf),'Always long spread')
flat=proxy_path(np.zeros(len(p)),'Always flat')
trading=pd.concat([long_short,always_long,flat],ignore_index=True)
trading_summary=trading.groupby('strategy').agg(total_proxy_usd_per_bbl=('net','sum'),active_months=('position',lambda q:int((q!=0).sum())))
display(trading_summary.round(3))
fig,ax=plt.subplots(figsize=(11,4))
for name,grp in trading.groupby('strategy'):
    ax.plot(grp['month'].astype(str),grp['index_100'],label=name)
ax.set(title='Illustrative spread strategy versus benchmarks',ylabel='100 + cumulative proxy USD/barrel',xlabel='Target month')
ax.set_xticks(range(0,len(p),12));ax.legend();plt.xticks(rotation=45);plt.tight_layout();plt.show()
print('CAUTION: baseline and OLS are proxy spread-change payoffs, not investable total returns.')

## 9. Robustness: longer assumed release delay and shorter estimation window
Robustness checks must not be used to select whichever variant performed best *after* inspecting the test. The main OLS4 validation choice remains frozen. Changing the release delay from two months to three requires reconstructing the affected features and eligible origins; a 120-month rolling window changes only the training sample.

In [ ]:
rolling120=rolling_predictions(frame,selected_ols,test_periods,window=120)
print('Expanding OLS:',score(test_panels[selected_ols]))
print('120-month OLS:',score(rolling120))
lag3=f.copy()
lag3['inventory_growth']=100*np.log(monthly['Inventories']).diff().shift(3)
lag3['ip_growth']=100*np.log(monthly['IndustrialProduction Index']).diff().shift(3)
lag3=lag3.dropna(subset=FEATURES+['target','spread_next','CL1','CL2'])
lag3_test=lag3.index[lag3.target_month>=TEST_START]
lag3_pred=rolling_predictions(lag3,selected_ols,lag3_test)
print('3-month delay OLS:',score(lag3_pred))
print('Different delayed-feature choices describe sensitivity; publication dates remain unverified.')

## 10. Interpretation and group presentation defence

**What we did:** We forecast next-month changes in the monthly WTI CL2–CL1 calendar spread using only the supplied futures, inventory and industrial-production observations. We compared three OLS specifications with zero-change and expanding historical-mean forecasts. The main specification was selected using 2015–2018 validation. An optional ARMA implementation is provided but is excluded from the default reported results.

**Forecast accuracy:** The validation-selected OLS4 model achieved an evaluation RMSE of approximately 0.7604 USD/barrel versus 0.7985 for zero change, a 9.31% reduction in MSE. However, MAE increased from 0.4598 to 0.4800. The model improves squared-error performance but does not improve average absolute accuracy. OLS2 performs better in the evaluation period; we retain the validation-selected OLS4 to avoid using evaluation results to choose the main specification.

**Uncertainty:** The 95% moving-block bootstrap interval for mean squared-loss improvement is approximately [−0.0404, 0.1919] and includes zero. The evidence does not establish a reliable forecasting advantage. The interval resamples existing forecast losses and does not account for model search or parameter re-estimation.

**Diagnostics:** Breusch–Godfrey does not reject no residual serial correlation at the tested lags. Breusch–Pagan and ARCH tests indicate heteroskedasticity and conditional variance dependence. HAC standard errors support more appropriate coefficient inference but do not remove these problems or improve forecasts. CUSUM does not reject parameter stability at 5%, but its p-value of about 0.071 warrants caution. The negative spread coefficient is consistent with conditional mean reversion; the other predictors are not individually significant at 5%. Predictor stationarity is assessed separately in Section 6. Statistical non-rejection is not proof that assumptions hold.

**Trading implications:** With the assumed threshold and costs, OLS4 generates a cumulative proxy payoff of 3.30 USD/barrel over 66 months. This is neither a percentage return nor verified executable futures profit. Continuous-contract differences do not fully represent held-contract returns; margin, collateral returns, variable bid–ask spreads and detailed roll implementation are excluded.

**Evaluation limitation:** The 2019–2024 period has previously been examined. These are later historical evaluation results, not a completely untouched final holdout. Further model changes require new observations or an appropriately designed additional evaluation. Publication delays are assumptions and cannot eliminate historical revision risk.

**Submission:** This notebook is one executable program file with an embedded, verified source-data extract. The coursework also requires a separate 2,000-word report and a five-minute presentation. This notebook does not replace either deliverable. Run **Kernel → Restart & Run All** before submission. If changing the data, settings or selected model, update this numerical interpretation to match the new outputs.
